In [0]:

import uuid
from datetime import datetime, timezone

from pyspark.sql import functions as F
from delta.tables import DeltaTable
from pyspark.sql.types import (
    StructType, StructField, StringType,
    TimestampType, LongType
)

# Configuration
RAW_PATH = "/Volumes/workspace/retail_training/raw_files"
TARGET_TABLE = "workspace.retail_training.bronze_sales"
BATCH_FILE = f"{RAW_PATH}/sales_batch_003.csv"

# Audit configuration
PIPELINE_NAME = "retail_data_pipeline"
TASK_NAME = "05_incremental_processing"
AUDIT_TABLE = "workspace.retail_training.pipeline_audit_log"

RUN_ID = str(uuid.uuid4())
START_TIME = datetime.now(timezone.utc)

# Track incoming rows processed during this task
ROWS_PROCESSED = 0

print(f"Task audit ID: {RUN_ID}")
print("Incremental processing configuration loaded.")


Task audit ID: f3bc4cce-b740-4bea-b0d7-359584e1097a
Incremental processing configuration loaded.


In [0]:

def write_audit_record(
    run_status,
    start_time,
    end_time,
    rows_processed=None,
    error_message=None
):
    audit_schema = StructType([
        StructField("pipeline_name", StringType(), False),
        StructField("task_name", StringType(), False),
        StructField("run_id", StringType(), True),
        StructField("run_status", StringType(), False),
        StructField("start_time", TimestampType(), True),
        StructField("end_time", TimestampType(), True),
        StructField("rows_processed", LongType(), True),
        StructField("error_message", StringType(), True),
        StructField("logged_at", TimestampType(), True)
    ])

    audit_record = [(
        PIPELINE_NAME,
        TASK_NAME,
        RUN_ID,
        run_status,
        start_time,
        end_time,
        rows_processed,
        error_message[:4000] if error_message else None,
        end_time
    )]

    audit_df = spark.createDataFrame(
        audit_record,
        schema=audit_schema
    )

    audit_df.write.format("delta").mode("append").saveAsTable(
        AUDIT_TABLE
    )


In [0]:

try:
    # 1. Check whether the optional batch file exists
    try:
        files = dbutils.fs.ls(RAW_PATH)

        batch_exists = any(
            file.name == "sales_batch_003.csv"
            for file in files
        )

    except Exception as e:
        raise RuntimeError(
            f"Unable to list raw files at {RAW_PATH}: {e}"
        ) from e

    # 2. Read the batch only when the file exists
    if batch_exists:
        batch = (
            spark.read
            .option("header", "true")
            .option("inferSchema", "true")
            .csv(BATCH_FILE)
            .withColumn("_ingested_at", F.current_timestamp())
        )

        incoming_count = batch.count()
        ROWS_PROCESSED = incoming_count

        print(f"Incoming batch rows: {incoming_count}")

        # 3. Reject records without a sale_id
        null_id_count = batch.filter(
            F.col("sale_id").isNull()
        ).count()

        if null_id_count > 0:
            raise ValueError(
                f"Batch contains {null_id_count} rows with null sale_id."
            )

        # 4. Remove duplicate IDs within the incoming batch
        deduplicated_batch = batch.dropDuplicates(["sale_id"])

        deduplicated_count = deduplicated_batch.count()
        duplicates_removed = incoming_count - deduplicated_count

        print(f"Duplicate rows removed: {duplicates_removed}")
        print(f"Rows after deduplication: {deduplicated_count}")

        # 5. Insert only sales that do not already exist
        target = DeltaTable.forName(spark, TARGET_TABLE)

        (
            target.alias("t")
            .merge(
                deduplicated_batch.alias("s"),
                "t.sale_id = s.sale_id"
            )
            .whenNotMatchedInsertAll()
            .execute()
        )

        print("Incremental merge completed.")

    else:
        ROWS_PROCESSED = 0
        print("No additional sales batch found. Incremental load skipped.")

    # 6. Validate the Bronze sales table
    target_metrics = spark.sql(f"""
        SELECT
            COUNT(*) AS total_rows,
            COUNT(DISTINCT sale_id) AS unique_sale_ids,
            SUM(
                CASE WHEN sale_id IS NULL THEN 1 ELSE 0 END
            ) AS null_sale_ids,
            MIN(sale_date) AS earliest_sale,
            MAX(sale_date) AS latest_sale
        FROM {TARGET_TABLE}
    """)

    display(target_metrics)

    metrics = target_metrics.first()

    total_rows = metrics["total_rows"]
    unique_ids = metrics["unique_sale_ids"]
    null_ids = metrics["null_sale_ids"] or 0

    if null_ids > 0 or total_rows != unique_ids:
        raise RuntimeError(
            "Bronze sales validation failed: "
            f"total_rows={total_rows}, "
            f"unique_sale_ids={unique_ids}, "
            f"null_sale_ids={null_ids}"
        )

    print("SUCCESS: Bronze sales has unique, non-null sale IDs.")
    print("Incremental processing notebook completed.")

except Exception as e:
    # Record the failure without hiding the original error
    end_time = datetime.now(timezone.utc)

    try:
        write_audit_record(
            run_status="FAILED",
            start_time=START_TIME,
            end_time=end_time,
            rows_processed=ROWS_PROCESSED,
            error_message=str(e)
        )
        print("FAILED audit record written.")
    except Exception as audit_error:
        print(f"WARNING: Could not write failure audit record: {audit_error}")

    raise

else:
    # Record successful processing and validation
    end_time = datetime.now(timezone.utc)

    write_audit_record(
        run_status="SUCCESS",
        start_time=START_TIME,
        end_time=end_time,
        rows_processed=ROWS_PROCESSED,
        error_message=None
    )

    print(f"SUCCESS audit record written: {ROWS_PROCESSED} input rows.")


Incoming batch rows: 5
Duplicate rows removed: 0
Rows after deduplication: 5
Incremental merge completed.


total_rows,unique_sale_ids,null_sale_ids,earliest_sale,latest_sale
110,110,0,2026-01-02,2026-04-05


SUCCESS: Bronze sales has unique, non-null sale IDs.
Incremental processing notebook completed.
SUCCESS audit record written: 5 input rows.
